
This Jupyter Notebook is part of a **collection of scripts curated by Tarlan Ahadli** for the **2025 Spring Semester**, based on **projects provided by Prof. Hajder Levente**. These scripts cover a range of computational topics, including **computer vision, image processing, numerical computing, and matrix transformations**, serving as educational resources for academic use.

### **Usage Restrictions**
This material is **strictly for academic and educational purposes**.  
**Commercial use is prohibited.**

For any **issues, suggestions, or inquiries**, please contact:  
📧 **tarlanahad@gmail.com**


# Affine and Perspective Transformations with OpenCV

This notebook provides a detailed explanation of affine and perspective transformations, including their mathematical proofs, Python implementations, and OpenCV demonstrations.

## 1. Understanding Transformations in Computer Vision

Transformations allow the manipulation of images by changing their position, shape, and perspective. These transformations are widely used in computer vision for image registration, object recognition, 3D reconstruction, and camera calibration.



## 2. Importing Required Libraries


In [7]:

import cv2
import numpy as np
import matplotlib.pyplot as plt

def show_image(title, img):
    """Utility function to display an image."""
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB) # Convert to RGB for correct visualization in matplotlib
    plt.figure(figsize=(6,6))
    plt.imshow(img_rgb)
    plt.title(title)
    plt.axis('off')
    plt.show()

# Load a sample image from OpenCV samples
img = cv2.imread('/content/apple.jpg')

if img is None:
    print("Could not load OpenCV sample image.")
else:
    show_image("Original Image", img)


Could not load OpenCV sample image.


[ WARN:0@1724.410] global loadsave.cpp:278 findDecoder imread_('/content/apple.jpg'): can't open/read file: check file path/integrity



## 3. Translation

### 3.1 Concept
Translation moves an image by shifting its pixel coordinates.

### 3.2 Proof

\begin{equation}
T =
\begin{bmatrix} 1 & 0 & t_x \\ 0 & 1 & t_y \\ 0 & 0 & 1 \end{bmatrix}
\end{equation}

Applying translation:

\begin{equation}
T \cdot \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} =
\begin{bmatrix} x + t_x \\ y + t_y \\ 1 \end{bmatrix}
\end{equation}

Thus, the coordinates move by \( (t_x, t_y) \).

### 3.3 Python Definition


In [8]:

def translate_image(img, tx, ty):
    """Applies translation to an image."""
    T = np.array([[1, 0, tx],
                  [0, 1, ty]], dtype=np.float32)
    return cv2.warpAffine(img, T, (img.shape[1], img.shape[0]))

# Example translation
translated_img = translate_image(img, 100, 50)
show_image("Translated Image", translated_img)


AttributeError: 'NoneType' object has no attribute 'shape'


## 4. Scaling

### 4.1 Concept
Scaling resizes an image by multiplying pixel coordinates.

### 4.2 Proof

\begin{equation}
S =
\begin{bmatrix} s_x & 0 & 0 \\ 0 & s_y & 0 \\ 0 & 0 & 1 \end{bmatrix}
\end{equation}

Applying scaling:

\begin{equation}
S \cdot \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} =
\begin{bmatrix} s_x \cdot x \\ s_y \cdot y \\ 1 \end{bmatrix}
\end{equation}

This scales the image by \( s_x, s_y \).

### 4.3 Python Definition


In [ ]:
def scale_image_warp(img, sx, sy):
    """Applies scaling using warpAffine and ensures correct output size."""
    T = np.array([[sx, 0, 0],
                  [0, sy, 0]], dtype=np.float32)

    new_size = (int(img.shape[1] * sx), int(img.shape[0] * sy))  # Adjusted output size

    return cv2.warpAffine(img, T, new_size)

# Example scaling
scaled_img_warp = scale_image_warp(img, 2.5, 2.5)
show_image("Scaled Image (warpAffine)", scaled_img_warp)

# Show new dimensions
print("Original Size:", img.shape)
print("Scaled Size (warpAffine):", scaled_img_warp.shape)



## 5. Rotation

### 5.1 Concept
Rotation rotates an image by a given angle around the origin.

### 5.2 Proof

\begin{equation}
R =
\begin{bmatrix} \cos \theta & -\sin \theta & 0 \\ \sin \theta & \cos \theta & 0 \\ 0 & 0 & 1 \end{bmatrix}
\end{equation}

Applying rotation:

\begin{equation}
R \cdot \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} =
\begin{bmatrix} x \cos \theta - y \sin \theta \\ x \sin \theta + y \cos \theta \\ 1 \end{bmatrix}
\end{equation}

### 5.3 Python Definition


In [ ]:

def rotate_image(img, angle):
    """Applies rotation to an image."""
    angle = np.deg2rad(angle)
    R = np.array([[np.cos(angle), -np.sin(angle), 0],
                  [np.sin(angle), np.cos(angle), 0]], dtype=np.float32)
    return cv2.warpAffine(img, R, (img.shape[1], img.shape[0]))

# Example rotation
rotated_img = rotate_image(img, 10)
show_image("Rotated Image", rotated_img)



## 6. Shearing (Skewing)

### 6.1 Concept
Shearing skews an image along an axis.

### 6.2 Proof

\begin{equation}
H =
\begin{bmatrix} 1 & k & 0 \\ 0 & 1 & 0 \\ 0 & 0 & 1 \end{bmatrix}
\end{equation}

Applying shearing:

\begin{equation}
H \cdot \begin{bmatrix} x \\ y \\ 1 \end{bmatrix} =
\begin{bmatrix} x + k y \\ y \\ 1 \end{bmatrix}
\end{equation}

### 6.3 Python Definition


In [ ]:

def shear_image(img, k):
    """Applies shearing to an image."""
    T = np.array([[1, k, 0],
                  [0, 1, 0]], dtype=np.float32)
    return cv2.warpAffine(img, T, (2*img.shape[1], 2*img.shape[0]))

# Example shearing
sheared_img = shear_image(img, 0.3)
show_image("Sheared Image", sheared_img)


### 7.2 Proof

A perspective transformation maps a 2D point \( (x, y) \) into a new coordinate system where parallel lines may converge. This is represented by a **homogeneous transformation matrix**:

$$
P =
\begin{bmatrix}
a & b & c \\
d & e & f \\
g & h & 1
\end{bmatrix}
$$

Applying this to a point in homogeneous coordinates:

$$
\begin{bmatrix}
x' \\
y' \\
w'
\end{bmatrix}
=
P
\begin{bmatrix}
x \\
y \\
1
\end{bmatrix}
$$

Expanding this multiplication:

$$
x' = ax + by + c
$$

$$
y' = dx + ey + f
$$

$$
w' = gx + hy + 1
$$

Since we work in **homogeneous coordinates**, the final pixel position is obtained by normalizing:

$$
x'_{\text{final}} = \frac{x'}{w'} = \frac{ax + by + c}{gx + hy + 1}
$$

$$
y'_{\text{final}} = \frac{y'}{w'} = \frac{dx + ey + f}{gx + hy + 1}
$$

This transformation ensures that **as \( w' \) varies**, the output coordinates experience a **perspective effect**, creating the appearance of depth.


In [ ]:

def perspective_transform(img):
    """Applies perspective transformation."""
    h, w = img.shape[:2]
    src_pts = np.float32([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]])
    dst_pts = np.float32([[50, 50], [w-50, 0], [0, h-50], [w, h]])
    T = cv2.getPerspectiveTransform(src_pts, dst_pts)
    return cv2.warpPerspective(img, T, (w, h))

# Example perspective transformation
perspective_img = perspective_transform(img)
show_image("Perspective Transformed Image", perspective_img)


## Combined Demonstration

In [ ]:
rotated_img = translate_image(rotate_image(img, 10), 40,-30)
show_image("Rotated Image", rotated_img)

In [ ]:
rotated_img = translate_image(img), 40,-30)

rotate_image(img, 10)
show_image("Rotated Image", rotated_img)

In [ ]:
import cv2
import numpy as np

def create_transformation(angle, tx, ty, sx, sy, skew_factor, persp1, persp2):
    """Creates a 3x3 transformation matrix with rotation, translation, scaling, skew, and perspective."""

    # Rotation Matrix
    R = np.eye(3, dtype=np.float32)
    R[0, 0] = np.cos(angle)
    R[0, 1] = -np.sin(angle)
    R[1, 0] = np.sin(angle)
    R[1, 1] = np.cos(angle)

    # Translation Matrix
    t = np.eye(3, dtype=np.float32)
    t[0, 2] = tx
    t[1, 2] = ty

    # Scaling Matrix
    Scale = np.eye(3, dtype=np.float32)
    Scale[0, 0] = sx
    Scale[1, 1] = sy

    # Skew Matrix
    Skew = np.eye(3, dtype=np.float32)
    Skew[0, 1] = skew_factor  # Shearing along x-axis

    # Combined Transformation
    T = t @ R @ Skew @ Scale  # Matrix multiplication


    # Perspective Transformation
    T[2, 0] = persp1
    T[2, 1] = persp2


    return T

def apply_transformation(img, T, is_perspective):
    """Applies transformation to the image using matrix multiplication."""

    height, width = img.shape[:2]
    new_img = np.zeros_like(img)

    for i in range(height):
        for j in range(width):
            # Homogeneous coordinates
            p = np.array([[i], [j], [1]], dtype=np.float32)

            # Apply transformation
            new_p = T @ p
            new_z = 1.0


            if is_perspective:  # Apply perspective division
                new_z = new_p[2, 0]


            new_x = int(new_p[0, 0] / new_z)
            new_y = int(new_p[1, 0] / new_z)

            # Check bounds before assigning pixels
            if 0 <= new_x < height and 0 <= new_y < width:
                new_img[new_x, new_y] = img[i, j]

    return new_img




# Create transformation matrix
T = create_transformation(angle=0.81, tx=200, ty=150, sx=0.5, sy=0.5, skew_factor=0.2, persp1=0.0001, persp2=0.002)
print("T =", T)

# Apply transformation
new_img = apply_transformation(img, T, is_perspective=True)

show_image("Transformations", new_img)

